# ST Score Restore — Stage 11 CameraPrIMuS Residual U-Net Training

Purpose: train the first **non-production** ST Restore Image Model baseline on paired CameraPrIMuS images stored in Google Drive.

Governance:
- `*_distorted.jpg` is the degraded input.
- `*.png` is the clean target.
- split is deterministic by source-family to prevent leakage.
- held-out data is never used for tuning.
- checkpoints and run evidence are written to Drive, not Git.
- no external pretrained weights are downloaded.
- this notebook does not authorize production inference, automatic final selection, or Stage 12.
- training remains fail-closed until the dataset rights review is explicitly recorded as `approved`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib, json, math, random, time

DATA_ROOT = Path('/content/drive/MyDrive/TEST/CameraPrIMuS/Corpus')
OUTPUT_ROOT = Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_OUTPUT/cameraprimus_residual_unet_v1')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 1107
IMAGE_SIZE = (128, 1024)  # H, W
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 2e-4

TRAINING_PURPOSE_AUTHORIZED = True
DATASET_RIGHTS_REVIEW_STATUS = 'review_required'  # change only after documented rights clearance
RUN_HELDOUT_FINAL_EVAL = False  # only after model/config are frozen

assert DATA_ROOT.exists(), f'CameraPrIMuS corpus not found: {DATA_ROOT}'
print('Data root:', DATA_ROOT)
print('Output root:', OUTPUT_ROOT)


In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def source_family_id(sample_id):
    return sample_id.split('_', 1)[0]

def split_for_family(family_id):
    bucket = int(hashlib.sha256(family_id.encode('utf-8')).hexdigest()[:8], 16) % 100
    if bucket < 80:
        return 'train'
    if bucket < 90:
        return 'development'
    return 'held_out'

pairs, rejected = [], []
for folder in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    sample_id = folder.name
    source = folder / f'{sample_id}_distorted.jpg'
    target = folder / f'{sample_id}.png'
    if not source.exists() or not target.exists():
        rejected.append({'sampleId': sample_id, 'reason': 'missing_source_or_target'})
        continue
    family = source_family_id(sample_id)
    pairs.append({
        'pairId': sample_id,
        'sourceFamilyId': family,
        'split': split_for_family(family),
        'sourcePath': str(source.relative_to(DATA_ROOT)),
        'targetPath': str(target.relative_to(DATA_ROOT)),
        'sourceSha256': sha256_file(source),
        'targetSha256': sha256_file(target),
        'agnosticPath': str((folder / f'{sample_id}.agnostic').relative_to(DATA_ROOT)) if (folder / f'{sample_id}.agnostic').exists() else None,
        'semanticPath': str((folder / f'{sample_id}.semantic').relative_to(DATA_ROOT)) if (folder / f'{sample_id}.semantic').exists() else None,
        'meiPath': str((folder / f'{sample_id}.mei').relative_to(DATA_ROOT)) if (folder / f'{sample_id}.mei').exists() else None,
    })

assert pairs, 'No valid CameraPrIMuS pairs discovered.'
family_splits = {}
for pair in pairs:
    previous = family_splits.setdefault(pair['sourceFamilyId'], pair['split'])
    assert previous == pair['split'], f"Source-family leakage: {pair['sourceFamilyId']}"

counts = {name: sum(p['split'] == name for p in pairs) for name in ('train', 'development', 'held_out')}
manifest = {
    'contractVersion': 'stage11.cameraprimus-paired-restoration.v1',
    'datasetRoot': 'TEST/CameraPrIMuS/Corpus',
    'pairCount': len(pairs),
    'rejectedCount': len(rejected),
    'splitCounts': counts,
    'sourceFamilySplitIsolation': True,
    'trainingPurposeAuthorized': TRAINING_PURPOSE_AUTHORIZED,
    'rightsReviewStatus': DATASET_RIGHTS_REVIEW_STATUS,
    'trainingExecutable': TRAINING_PURPOSE_AUTHORIZED and DATASET_RIGHTS_REVIEW_STATUS == 'approved',
    'pairs': pairs,
    'rejected': rejected,
}
manifest_path = OUTPUT_ROOT / 'cameraprimus_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps({k: manifest[k] for k in ('pairCount','rejectedCount','splitCounts','trainingExecutable')}, indent=2))
print('Manifest:', manifest_path)


In [ ]:
if DATASET_RIGHTS_REVIEW_STATUS != 'approved':
    raise RuntimeError(
        'Training is intentionally blocked: CameraPrIMuS rights review is not recorded as approved. '
        'The manifest is ready; record rights clearance before running the training cells.'
    )


In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import functional as TF

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'GPU runtime required for the intended Stage 11 run.'


In [ ]:
class PairedScoreDataset(Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        record = self.records[idx]
        source = Image.open(DATA_ROOT / record['sourcePath']).convert('L').resize((IMAGE_SIZE[1], IMAGE_SIZE[0]), Image.Resampling.BILINEAR)
        target = Image.open(DATA_ROOT / record['targetPath']).convert('L').resize((IMAGE_SIZE[1], IMAGE_SIZE[0]), Image.Resampling.BILINEAR)
        return TF.pil_to_tensor(source).float().div(255.0), TF.pil_to_tensor(target).float().div(255.0)

def loader_for(split, shuffle):
    records = [p for p in pairs if p['split'] == split]
    return DataLoader(PairedScoreDataset(records), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = loader_for('train', True)
dev_loader = loader_for('development', False)
heldout_loader = loader_for('held_out', False)
print('Batches:', len(train_loader), len(dev_loader), len(heldout_loader))


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class ResidualUNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.e1 = DoubleConv(1, base)
        self.e2 = DoubleConv(base, base*2)
        self.e3 = DoubleConv(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.b = DoubleConv(base*4, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.d3 = DoubleConv(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.d2 = DoubleConv(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.d1 = DoubleConv(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        b = self.b(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        residual = torch.tanh(self.out(d1)) * 0.5
        return torch.clamp(x + residual, 0.0, 1.0)

def gradient_map(x):
    gx = x[:, :, :, 1:] - x[:, :, :, :-1]
    gy = x[:, :, 1:, :] - x[:, :, :-1, :]
    return gx, gy

def restoration_loss(pred, target):
    pixel = torch.mean(torch.abs(pred - target))
    pgx, pgy = gradient_map(pred)
    tgx, tgy = gradient_map(target)
    edge = torch.mean(torch.abs(pgx - tgx)) + torch.mean(torch.abs(pgy - tgy))
    return pixel + 0.25 * edge

def batch_psnr(pred, target):
    mse = torch.mean((pred - target) ** 2).clamp_min(1e-12)
    return float(10.0 * torch.log10(1.0 / mse))


In [ ]:
model = ResidualUNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
last_checkpoint = OUTPUT_ROOT / 'last.pt'
best_checkpoint = OUTPUT_ROOT / 'best.pt'
start_epoch, best_dev = 0, float('inf')

if last_checkpoint.exists():
    ckpt = torch.load(last_checkpoint, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    best_dev = ckpt['bestDevLoss']
    print('Resumed from epoch', start_epoch)

history = []
for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_sum, train_n = 0.0, 0
    for source, target in train_loader:
        source, target = source.to(device, non_blocking=True), target.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        pred = model(source)
        loss = restoration_loss(pred, target)
        loss.backward()
        optimizer.step()
        train_sum += float(loss) * source.size(0)
        train_n += source.size(0)

    model.eval()
    dev_sum, dev_n, psnr_sum = 0.0, 0, 0.0
    with torch.no_grad():
        for source, target in dev_loader:
            source, target = source.to(device, non_blocking=True), target.to(device, non_blocking=True)
            pred = model(source)
            loss = restoration_loss(pred, target)
            dev_sum += float(loss) * source.size(0)
            dev_n += source.size(0)
            psnr_sum += batch_psnr(pred, target) * source.size(0)

    train_loss = train_sum / max(train_n, 1)
    dev_loss = dev_sum / max(dev_n, 1)
    dev_psnr = psnr_sum / max(dev_n, 1)
    row = {'epoch': epoch, 'trainLoss': train_loss, 'developmentLoss': dev_loss, 'developmentPSNR': dev_psnr}
    history.append(row)
    print(row)

    payload = {
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'bestDevLoss': min(best_dev, dev_loss),
        'config': {'seed': SEED, 'imageSize': IMAGE_SIZE, 'batchSize': BATCH_SIZE, 'learningRate': LEARNING_RATE},
    }
    torch.save(payload, last_checkpoint)
    if dev_loss < best_dev:
        best_dev = dev_loss
        torch.save(payload, best_checkpoint)

(OUTPUT_ROOT / 'training_history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
print('Best development loss:', best_dev)


In [ ]:
# Final held-out evaluation is deliberately separate from training/tuning.
if RUN_HELDOUT_FINAL_EVAL:
    assert best_checkpoint.exists()
    frozen = torch.load(best_checkpoint, map_location=device)
    model.load_state_dict(frozen['model'])
    model.eval()
    loss_sum = psnr_sum = n = 0.0
    with torch.no_grad():
        for source, target in heldout_loader:
            source, target = source.to(device), target.to(device)
            pred = model(source)
            loss_sum += float(restoration_loss(pred, target)) * source.size(0)
            psnr_sum += batch_psnr(pred, target) * source.size(0)
            n += source.size(0)
    heldout = {'loss': loss_sum/max(n,1), 'psnr': psnr_sum/max(n,1), 'count': int(n)}
    (OUTPUT_ROOT / 'heldout_final_metrics.json').write_text(json.dumps(heldout, indent=2), encoding='utf-8')
    print(heldout)
else:
    print('Held-out evaluation not run. Freeze the model/config first, then set RUN_HELDOUT_FINAL_EVAL=True.')


In [ ]:
run_evidence = {
    'stage': 11,
    'architecture': 'residual_unet',
    'dataset': 'CameraPrIMuS',
    'manifestPath': str(manifest_path),
    'checkpointPath': str(best_checkpoint),
    'trainingPurposeAuthorized': TRAINING_PURPOSE_AUTHORIZED,
    'rightsReviewStatus': DATASET_RIGHTS_REVIEW_STATUS,
    'productionInferenceAuthorized': False,
    'automaticFinalSelectionAuthorized': False,
    'stage12EntryAuthorized': False,
    'stage9aPreservationEvaluationPending': True,
    'stage9ComparatorPending': True,
    'stage10SelectorPending': True,
}
(OUTPUT_ROOT / 'run_evidence.json').write_text(json.dumps(run_evidence, indent=2), encoding='utf-8')
print(json.dumps(run_evidence, indent=2))
